# Evaluate the baseline Go1 policy

Loads the baseline policy saved to Google Drive (trained in `locomotion.ipynb`, ideal motors) and measures the reference numbers that every motor-model experiment is compared against:

- **Fall rate** and **survival time** (out of a 20 s episode)
- **Velocity-tracking error**: forward/sideways speed and turning rate vs. the commanded values
- **Peak joint torque**

It also renders a short, small GIF for the README and outreach emails.

**Runtime:** GPU (Runtime → Change runtime type → T4 GPU).

## 1. Install (run once, then restart)

After this cell finishes, go to **Runtime → Restart session**, then continue from step 2. If Colab asks to restart again later because of Brax, click **Cancel**.

In [2]:
# Same fixes as the training notebook: Colab's JAX is newer than the latest Flax/Brax releases expect.
!pip install -q mujoco mujoco_mjx playground==0.2.0 mediapy
!pip install -q -U flax
!pip install -q --force-reinstall --no-deps git+https://github.com/google/brax.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Setup

In [6]:
%env MUJOCO_GL=egl

import json, os
import jax
import jax.numpy as jp
import numpy as np
import mediapy as media
from google.colab import drive

from brax.io import model
from brax.training.acme import running_statistics
from brax.training.agents.ppo import networks as ppo_networks
from mujoco_playground import registry
from mujoco_playground.config import locomotion_params

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/motor-rl/baseline_go1'
print('JAX backend:', jax.default_backend())  # should print: gpu

env: MUJOCO_GL=egl
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
JAX backend: gpu


## 3. Load the environment and the saved policy

In [8]:
ENV_NAME = 'Go1JoystickFlatTerrain'
env_cfg = registry.get_default_config(ENV_NAME)
env_cfg.impl = 'jax'                      # Warp isn't used in this project
env = registry.load(ENV_NAME, config=env_cfg)
ppo_params = locomotion_params.brax_ppo_config(ENV_NAME)

params = model.load_params(f'{OUT}/params')

# Rebuild the same network shape used in training, then plug in the saved weights.
ppo_net = ppo_networks.make_ppo_networks(
    env.observation_size,
    env.action_size,
    preprocess_observations_fn=running_statistics.normalize,
    **ppo_params.network_factory,
)
policy = ppo_networks.make_inference_fn(ppo_net)(params, deterministic=True)
print('Policy loaded. Control step:', env.dt, 's')

Policy loaded. Control step: 0.02 s


## 4. Measure the baseline

Runs 256 robots in parallel for one full episode (1,000 control steps = 20 s each). Commands are sampled by the environment, so a fixed random seed gives every experiment the same command sequences.

A robot counts as fallen when the environment ends its episode (body tipped over). Errors and torques are only averaged over the time each robot is still standing.

In [9]:
N_ROBOTS = 256
N_STEPS = 1000
SEED = 0   # keep this fixed across experiments so they see identical commands

def run_one(key):
    state = env.reset(key)

    def step(carry, _):
        state, alive, key = carry
        key, act_key = jax.random.split(key)
        action, _ = policy(state.obs, act_key)
        state = env.step(state, action)

        cmd = state.info['command']               # [forward m/s, sideways m/s, yaw rad/s]
        vel = env.get_local_linvel(state.data)    # body-frame linear velocity
        gyro = env.get_gyro(state.data)           # body-frame angular velocity
        lin_err = jp.linalg.norm(vel[:2] - cmd[:2])
        yaw_err = jp.abs(gyro[2] - cmd[2])
        peak_tau = jp.max(jp.abs(state.data.actuator_force))

        next_alive = alive * (1.0 - state.done)
        return (state, next_alive, key), (alive, lin_err, yaw_err, peak_tau)

    _, out = jax.lax.scan(step, (state, jp.float32(1.0), key), None, length=N_STEPS)
    return out

keys = jax.random.split(jax.random.PRNGKey(SEED), N_ROBOTS)
alive, lin_err, yaw_err, peak_tau = jax.jit(jax.vmap(run_one))(keys)   # first run compiles (~1-2 min)
alive, lin_err, yaw_err, peak_tau = map(np.asarray, (alive, lin_err, yaw_err, peak_tau))

In [10]:
standing = alive.sum()
results = {
    'run': 'baseline_ideal_motors',
    'n_robots': N_ROBOTS,
    'episode_s': N_STEPS * env.dt,
    'fall_rate': float(1.0 - alive[:, -1].mean()),
    'mean_survival_s': float(alive.sum(axis=1).mean() * env.dt),
    'lin_vel_error_mps': float((lin_err * alive).sum() / standing),
    'yaw_rate_error_radps': float((yaw_err * alive).sum() / standing),
    'peak_torque_p95_Nm': float(np.percentile(peak_tau[alive > 0], 95)),
    'peak_torque_max_Nm': float(peak_tau[alive > 0].max()),
    'seed': SEED,
}
for k, v in results.items():
    print(f'{k:>24}: {v}')

with open(f'{OUT}/eval_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved to', f'{OUT}/eval_metrics.json')

                     run: baseline_ideal_motors
                n_robots: 256
               episode_s: 20.0
               fall_rate: 0.00390625
         mean_survival_s: 19.92445182800293
       lin_vel_error_mps: 0.1013055145740509
    yaw_rate_error_radps: 0.07145024836063385
      peak_torque_p95_Nm: 15.262296676635742
      peak_torque_max_Nm: 30.345733642578125
                    seed: 0
Saved to /content/drive/MyDrive/motor-rl/baseline_go1/eval_metrics.json


## 5. Small GIF for the README and emails

5 seconds of walking forward at 1 m/s, 320×240, every other frame. Should come out at a few MB instead of 15 MB.

In [11]:
COMMAND = jp.array([1.0, 0.0, 0.0])   # walk forward at 1 m/s
CLIP_STEPS = 250                      # 250 x 0.02 s = 5 s

reset_fn, step_fn, pol_fn = jax.jit(env.reset), jax.jit(env.step), jax.jit(policy)
state = reset_fn(jax.random.PRNGKey(1))
key = jax.random.PRNGKey(2)
traj = [state]
for _ in range(CLIP_STEPS):
    state.info['command'] = COMMAND
    key, k = jax.random.split(key)
    action, _ = pol_fn(state.obs, k)
    state = step_fn(state, action)
    traj.append(state)

frames = env.render(traj[::2], camera='track', width=320, height=240)
fps = 1.0 / env.dt / 2
media.show_video(frames, fps=fps)

gif_path = f'{OUT}/go1_baseline_small.gif'
media.write_video(gif_path, frames, fps=fps, codec='gif')
print('GIF size: %.1f MB' % (os.path.getsize(gif_path) / 1e6))

100%|██████████| 126/126 [01:46<00:00,  1.19it/s]


This browser does not support the video tag.


GIF size: 3.1 MB


---
Simulation, robot model and training setup from [MuJoCo Playground](https://github.com/google-deepmind/mujoco_playground) (Apache 2.0).